<img src="figs/RNN_fig1.png" alt="альтернативный_текст">


# Прямой проход

In [1]:

import torch

import torch.nn as nn

In [2]:
input_size = 20  # размерность входного вектора
hidden_size = 10  # размерность скрытого состояния
num_layers = 1   # количество рекуррентных слоёв

In [3]:
rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
rnn

RNN(20, 10, batch_first=True)

In [4]:
batch_size = 2
seq_length = 5  # количество токенов в каждой последовательности


# Создадим случайный входной тензор
# батч из двух последовательностей по пять токенов
# где каждый токен кодируется вектором из 20 признаков
x = torch.randn(batch_size, seq_length, input_size)
x.shape

torch.Size([2, 5, 20])

In [5]:

# Инициализация начального скрытого состояния для каждого слоя
# и каждой последовательности
h0 = torch.zeros(num_layers, batch_size, hidden_size)
h0.shape

torch.Size([1, 2, 10])

In [6]:
# Прямой проход
output, hn = rnn(x, h0)
output.shape, hn.shape

(torch.Size([2, 5, 10]), torch.Size([1, 2, 10]))

<img src="figs/RNN_fig2.png" alt="альтернативный_текст">


In [7]:

# вывод результатов
print("Выходной тензор (output):", output.shape)
print(output)


print("Последнее скрытое состояние (hn):", hn.shape)
print(hn)

Выходной тензор (output): torch.Size([2, 5, 10])
tensor([[[ 0.7628, -0.8653, -0.1888,  0.3263,  0.3260, -0.3084,  0.5752,
           0.2695,  0.6938,  0.5017],
         [ 0.9804, -0.6767, -0.1113, -0.4881, -0.2818,  0.5992,  0.3575,
           0.8545,  0.9427, -0.6419],
         [ 0.8304, -0.4826,  0.4321,  0.5784,  0.2421,  0.5730, -0.1593,
           0.8187,  0.1897,  0.9538],
         [ 0.5871,  0.3017,  0.7126,  0.1143, -0.1681,  0.7253, -0.1764,
           0.1133,  0.7720,  0.8794],
         [ 0.9464, -0.7103,  0.7959,  0.9087,  0.9739,  0.4904,  0.5063,
          -0.1123, -0.4479,  0.0258]],

        [[ 0.2458, -0.8170, -0.8633, -0.0134,  0.7274, -0.6965,  0.4867,
           0.2161,  0.3947,  0.3971],
         [-0.1657,  0.6882,  0.8851, -0.4719,  0.8689, -0.4487, -0.7453,
           0.4887, -0.7912,  0.4960],
         [ 0.6997, -0.9423, -0.9374,  0.0413, -0.0860, -0.5396,  0.9750,
          -0.2485,  0.6770,  0.4255],
         [ 0.9197,  0.7789,  0.5175,  0.8097, -0.2079, -0.644

# Embedding
Использовать эмбеддинг-слой можно так:

<img src="figs/Embedding_fig1.png" alt="альтернативный_текст">

In [8]:
# создаём эмбеддинг-слой
emb_layer = nn.Embedding(vocab_size, embedding_dim)

# создаём RNN-слой
rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)

# считаем эмбеддинг-слой
emb_layer_res = emb_layer(input_sequence)

# считаем RNN-слой
rnn_layer_res = rnn(emb_layer_res)

NameError: name 'vocab_size' is not defined

# Padding & masking
для рекуррентных сетей вручную считать маски не нужно. В `PyTorch` уже реализована функция `pack_padded_sequence`, которая принимает на вход тексты после пэддинга и их длины и возвращает объект типа `PackedSequence`. Этот объект можно передать на вход нейросети при forward-проходе. А потом можно распаковать выход функцией `pad_packed_sequence` — на векторы и изначальные длины. Это нужно, чтобы RNN для каждой последовательности не проходила по токенам пэддинга.

In [9]:

from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


# Предположим, что длины последовательностей известны:
lengths = torch.tensor([5, 3, 2])


# Упаковываем последовательности перед подачей в RNN

packed_x = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)


# Прямой проход с упакованными последовательностями
packed_output, hn = rnn(packed_x, h0)


# Распаковываем обратно выходы
output, output_lengths = pad_packed_sequence(packed_output, batch_first=True)

IndexError: index out of range in self